In [1]:

# 03test_feature_engeneering.py
import pandas as pd
import numpy as np
import os

# --- 1. 定義とパス設定 ---
PREDICT_DIR = '../data/processed_test/'
cleaned_data_path = os.path.join(PREDICT_DIR, 'test_cleaned.parquet')
processed_file = 'test_features.parquet'
processed_path = os.path.join(PREDICT_DIR, processed_file)


# --- 2. データの読み込み ---
print("2. クレンジング済みデータを読み込みます。")

df_test = pd.read_parquet(cleaned_data_path)
features_data_path = '../data/processed/features.parquet'
df = pd.read_parquet(features_data_path)

2. クレンジング済みデータを読み込みます。


In [2]:
    df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19466 entries, 0 to 19465
Data columns (total 19 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   ID         19466 non-null  int64  
 1   市区町村コード    19466 non-null  int64  
 2   都道府県名      19466 non-null  object 
 3   市区町村名      19466 non-null  object 
 4   地区名        19463 non-null  object 
 5   最寄駅：名称     19453 non-null  object 
 6   最寄駅：距離（分）  18509 non-null  float64
 7   間取り        18544 non-null  object 
 8   面積（㎡）      19466 non-null  int64  
 9   建築年        18804 non-null  object 
 10  建物の構造      18201 non-null  object 
 11  用途         13480 non-null  object 
 12  今後の利用目的    18439 non-null  object 
 13  都市計画       19122 non-null  object 
 14  建ぺい率（％）    19045 non-null  float64
 15  容積率（％）     19045 non-null  float64
 16  取引時点       19466 non-null  object 
 17  改装         17036 non-null  object 
 18  取引の事情等     152 non-null    object 
dtypes: float64(3), int64(3), object(13)
memory usa

#　既存データから特徴量作成

In [2]:

# '建築年'の処理関数
def convert_wareki_to_seireki(wareki):
    if pd.isna(wareki) or wareki is None:
        return np.nan
    
    wareki = str(wareki).strip()
    
    # '戦前'を1945に変換
    if wareki == '戦前':
        return 1945.0
        
    # 和暦変換ロジック
    try:
        if '昭和' in wareki: return 1925 + int(wareki.replace('昭和', '').replace('年', '').replace('元', '1'))
        elif '平成' in wareki: return 1988 + int(wareki.replace('平成', '').replace('年', '').replace('元', '1'))
        elif '令和' in wareki: return 2018 + int(wareki.replace('令和', '').replace('年', '').replace('元', '1'))
        elif '大正' in wareki: return 1911 + int(wareki.replace('大正', '').replace('年', '').replace('元', '1'))
        elif '明治' in wareki: return 1867 + int(wareki.replace('明治', '').replace('年', '').replace('元', '1'))
        # 西暦の場合はそのまま返す (元のデータ型を float に統一)
        return float(wareki)
    except:
        return np.nan # 変換できないものは NaN

# '取引時点'の処理
if '取引時点' in df_test.columns:
    # 四半期情報を使わず、年の数値のみを抽出
    df_test['取引時点_年'] = df_test['取引時点'].str.extract(r'(\d{4})').astype(float)
else:
    df_test['取引時点_年'] = np.nan

# '建築年'を西暦に変換
if '建築年' in df_test.columns:
    df_test['建築年_西暦'] = df_test['建築年'].apply(convert_wareki_to_seireki)
else:
    df_test['建築年_西暦'] = np.nan

# 1. 築年数_欠損フラグの追加
df_test['築年数_欠損'] = df_test['建築年_西暦'].isna().astype(int)

# 2. 取引時点での築年数の追加 (NaNはNaNのまま保持)
df_test['取引時点での築年数'] = df_test['取引時点_年'] - df_test['建築年_西暦']

In [4]:
df_test['取引の事情等'].unique()

array([None, '調停・競売等', '関係者間取引', '瑕疵有りの可能性', 'その他事情有り',
       '他の権利・負担付き、調停・競売等'], dtype=object)

In [3]:
# 03test_feature_engeneering.py (取引事情ダミー変数作成部分)

# ------------------------------------------------------------------------------
# 3.4. 取引事情ダミー変数の作成と統合 (モデル要求の3列を生成)
# ------------------------------------------------------------------------------
print("3.4. 取引事情ダミー変数を作成・統合します。")

# 1. 欠損値の処理: 訓練時と同様に、欠損値 (None) を明示的なカテゴリとして扱う
# NaNを 'None' という文字列に置き換える。これにより、get_dummiesで一つのカテゴリとして扱われる。
ORIGINAL_COL = '取引の事情等'
if ORIGINAL_COL in df_test.columns:
    df_test[ORIGINAL_COL] = df_test[ORIGINAL_COL].fillna('None')
else:
    print(f"   ⚠️ 警告: 元の列 '{ORIGINAL_COL}' が df_test に存在しません。処理をスキップします。")
    # この後の処理で KeyError を避けるため、必要な3列を0で初期化して終了する
    df_test['取引の事情等_関係者間取引'] = 0
    df_test['取引の事情等_調停・競売等'] = 0
    df_test['取引の事情等_その他'] = 0
    # return # 実際にはここで関数を抜ける

    
# 2. 訓練データで確認された全てのカテゴリをリスト化
# ⚠️ 訓練データで確認された全てのカテゴリを正確に含めてください。
KNOWN_CATEGORIES_FROM_TRAIN = [
    'None', 
    '調停・競売等', 
    '関係者間取引', 
    '瑕疵有りの可能性', 
    'その他事情有り',
    '他の権利・負担付き、調停・競売等'
]

# 3. 訓練データと同じダミー変数のセットを生成
# reindexを利用して、テストデータに存在しないカテゴリ列を0で埋めるのが最も安全。
# 元の列をダミー変数に変換
df_dummies = pd.get_dummies(df_test[ORIGINAL_COL], prefix='取引の事情等', dtype=int)

# 訓練データで存在した可能性のある全ての列名を定義
required_dummy_cols_all = [f'取引の事情等_{cat}' for cat in KNOWN_CATEGORIES_FROM_TRAIN]

# reindexを使用して、不足している列を0で埋める
df_dummies = df_dummies.reindex(columns=required_dummy_cols_all, fill_value=0)

# 4. モデルが要求する最終的な3列に統合

# A) 取引の事情等_調停・競売等: '調停・競売等' のダミー列
df_test['取引の事情等_調停・競売等'] = df_dummies['取引の事情等_調停・競売等']

# B) 取引の事情等_関係者間取引: '関係者間取引' のダミー列
df_test['取引の事情等_関係者間取引'] = df_dummies['取引の事情等_関係者間取引']


# C) 取引の事情等_その他: 残りの複雑な事情を合計
# 'その他' に含めるカテゴリ（訓練時の定義に基づく）
other_categories = [
    '瑕疵有りの可能性', 
    'その他事情有り',
    '他の権利・負担付き、調停・競売等'
]
other_cols = [f'取引の事情等_{cat}' for cat in other_categories]

# 合計列を作成し、df_testに追加
df_test['取引の事情等_その他'] = df_dummies[other_cols].sum(axis=1)


# 5. 中間ダミー列のクリーンアップ（オプション）
# 元の df_test に結合した中間ダミー列を削除（メモリ効率のため）
# df_test = df_test.drop(columns=required_dummy_cols_all, errors='ignore')

print("   ✅ 取引事情ダミー変数3列の作成と統合が完了しました。")

3.4. 取引事情ダミー変数を作成・統合します。
   ✅ 取引事情ダミー変数3列の作成と統合が完了しました。


In [ ]:
df_test

In [4]:
df_test=pd.get_dummies(df_test,columns=["改装"],dtype=int)
df_test

,ID,市区町村コード,都道府県名,市区町村名,地区名,最寄駅：名称,最寄駅：距離（分）,間取り,面積（㎡）,建築年,...,取引の事情等,取引時点_年,建築年_西暦,築年数_欠損,取引時点での築年数,取引の事情等_調停・競売等,取引の事情等_関係者間取引,取引の事情等_その他,改装_改装済,改装_未改装
0,1000000,1101,北海道,札幌市中央区,旭ケ丘,円山公園,26.0,３ＬＤＫ,75,昭和64年,...,None,2020.0,1989.0,0,31.0,0,0,0,0,1
1,1000056,1101,北海道,札幌市中央区,大通西,西１１丁目,1.0,２ＬＤＫ,55,平成28年,...,None,2020.0,2016.0,0,4.0,0,0,0,0,1
2,1000108,1101,北海道,札幌市中央区,大通西,西１８丁目,2.0,１Ｒ,15,昭和64年,...,None,2020.0,1989.0,0,31.0,0,0,0,0,1
3,1000109,1101,北海道,札幌市中央区,大通西,西１８丁目,2.0,１ＬＤＫ,45,平成3年,...,None,2020.0,1991.0,0,29.0,0,0,0,1,0
4,1000110,1101,北海道,札幌市中央区,大通西,西１８丁目,3.0,１Ｒ,20,昭和56年,...,None,2020.0,1981.0,0,39.0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19461,47003828,47208,沖縄県,浦添市,牧港,None,NaN,４ＬＤＫ,80,平成31年,...,None,2020.0,2019.0,0,1.0,0,0,0,0,1
19462,47003829,47208,沖縄県,浦添市,牧港,None,NaN,２ＬＤＫ,70,平成10年,...,None,2020.0,1998.0,0,22.0,0,0,0,1,0
19463,47003880,47208,沖縄県,浦添市,港川,None,NaN,４ＬＤＫ,50,平成12年,...,None,2020.0,2000.0,0,20.0,0,0,0,0,1
19464,47006648,47211,沖縄県,沖縄市,与儀,None,NaN,３ＬＤＫ,60,平成31年,...,None,2020.0,2019.0,0,1.0,0,0,0,0,1


In [5]:
# 1. 欠損値を一時的に埋める
df_test["間取り_filled"] = df_test["間取り"].fillna("欠損値")

# 2. 頻度を計算する (新しい'間取り_filled'列に対して)
freq = df_test["間取り_filled"].value_counts()

# 3. グループ化の適用
df_test["間取り_grouped"] = df_test["間取り_filled"].apply(lambda x: x if freq[x] > 1000 else "その他")

# # 3. ダミー変数化（0/1化）
df_test = pd.get_dummies(df_test, columns=["間取り_grouped"], dtype=int)

df_test.head()

,ID,市区町村コード,都道府県名,市区町村名,地区名,最寄駅：名称,最寄駅：距離（分）,間取り,面積（㎡）,建築年,...,取引の事情等_その他,改装_改装済,改装_未改装,間取り_filled,間取り_grouped_その他,間取り_grouped_１Ｋ,間取り_grouped_１ＬＤＫ,間取り_grouped_２ＬＤＫ,間取り_grouped_３ＬＤＫ,間取り_grouped_４ＬＤＫ
0,1000000,1101,北海道,札幌市中央区,旭ケ丘,円山公園,26.0,３ＬＤＫ,75,昭和64年,...,0,0,1,３ＬＤＫ,0,0,0,0,1,0
1,1000056,1101,北海道,札幌市中央区,大通西,西１１丁目,1.0,２ＬＤＫ,55,平成28年,...,0,0,1,２ＬＤＫ,0,0,0,1,0,0
2,1000108,1101,北海道,札幌市中央区,大通西,西１８丁目,2.0,１Ｒ,15,昭和64年,...,0,0,1,１Ｒ,1,0,0,0,0,0
3,1000109,1101,北海道,札幌市中央区,大通西,西１８丁目,2.0,１ＬＤＫ,45,平成3年,...,0,1,0,１ＬＤＫ,0,0,1,0,0,0
4,1000110,1101,北海道,札幌市中央区,大通西,西１８丁目,3.0,１Ｒ,20,昭和56年,...,0,0,0,１Ｒ,1,0,0,0,0,0


In [6]:
df_test = df_test.drop(columns=["間取り_filled"])

In [16]:
df_city=pd.get_dummies(df_test, columns=['都市計画'], dtype=int)
df_test['都市計画_高価格帯'] = (
    df_city['都市計画_第１種低層住居専用地域'] +
    df_city['都市計画_第２種低層住居専用地域'] +
    df_city['都市計画_工業専用地域'] 
)

df_test['都市計画_中価格帯'] = (
    df_city['都市計画_商業地域'] + 
    df_city['都市計画_準工業地域'] +  
    df_city['都市計画_工業地域'] +  
    df_city['都市計画_第１種中高層住居専用地域'] +
    df_city['都市計画_第２種中高層住居専用地域'] +
    df_city['都市計画_第１種住居地域'] +
    df_city['都市計画_第２種住居地域'] +
    df_city['都市計画_準住居地域'] +
    df_city['都市計画_近隣商業地域'] 
)

df_test['都市計画_低価格帯'] = (
    df_city['都市計画_市街化調整区域'] +
    df_city['都市計画_市街化区域及び市街化調整区域外の都市計画区域'] 
)

In [7]:
import re
# 新しく追加する人口密度のテキストデータ
density_text = """
1東京都6,451.17
2大阪府4,603.02
3神奈川県3,816.89
4埼玉県1,929.89
5愛知県1,443.06
6千葉県1,217.00
7福岡県1,022.06
8沖縄県642.85
9兵庫県635.26
10京都府546.65
11香川県488.61
12茨城県460.79
13静岡県453.15
14滋賀県348.69
15奈良県348.18
16佐賀県322.73
17広島県320.44
18宮城県308.58
19長崎県302.75
20群馬県296.97
21三重県296.37
22栃木県293.74
23石川県262.42
24岡山県257.31
25富山県234.48
26熊本県228.92
27愛媛県224.70
28山口県209.32
29和歌山県186.18
30岐阜県180.12
31山梨県176.97
32福井県176.27
33大分県171.15
34新潟県166.79
35鹿児島県166.58
36徳島県165.27
37鳥取県151.43
38長野県146.62
39宮崎県133.35
40福島県126.40
41青森県120.76
42山形県108.42
43島根県95.62
44高知県92.32
45秋田県77.02
46岩手県74.92
47北海道64.29
"""

# --- Step 2: 人口密度データを整形してデータフレームを作成 ---

prefectures_density = []
densities = []

# テキストを行ごとに分割して処理
for line in density_text.strip().split('\n'):
    # 正規表現で都道府県名と数値を抽出
    match = re.search(r'\d+([^\d,.]+)\s*([\d,.]+)', line)
    if match:
        prefecture = match.group(1).strip()
        density_str = match.group(2)

        # コンマを削除して浮動小数点数に変換
        density_float = float(density_str.replace(',', ''))

        prefectures_density.append(prefecture)
        densities.append(density_float)

# 人口密度のデータフレームを作成
density_df = pd.DataFrame({
    '都道府県名': prefectures_density,
    '人口密度': densities
})


# --- Step 3: 最終的な結合 ---

# 坪単価が入ったdfに、さらに人口密度のデータを結合
df_test = pd.merge(df_test, density_df, on='都道府県名', how='left')
df_test


,ID,市区町村コード,都道府県名,市区町村名,地区名,最寄駅：名称,最寄駅：距離（分）,間取り,面積（㎡）,建築年,...,取引の事情等_その他,改装_改装済,改装_未改装,間取り_grouped_その他,間取り_grouped_１Ｋ,間取り_grouped_１ＬＤＫ,間取り_grouped_２ＬＤＫ,間取り_grouped_３ＬＤＫ,間取り_grouped_４ＬＤＫ,人口密度
0,1000000,1101,北海道,札幌市中央区,旭ケ丘,円山公園,26.0,３ＬＤＫ,75,昭和64年,...,0,0,1,0,0,0,0,1,0,64.29
1,1000056,1101,北海道,札幌市中央区,大通西,西１１丁目,1.0,２ＬＤＫ,55,平成28年,...,0,0,1,0,0,0,1,0,0,64.29
2,1000108,1101,北海道,札幌市中央区,大通西,西１８丁目,2.0,１Ｒ,15,昭和64年,...,0,0,1,1,0,0,0,0,0,64.29
3,1000109,1101,北海道,札幌市中央区,大通西,西１８丁目,2.0,１ＬＤＫ,45,平成3年,...,0,1,0,0,0,1,0,0,0,64.29
4,1000110,1101,北海道,札幌市中央区,大通西,西１８丁目,3.0,１Ｒ,20,昭和56年,...,0,0,0,1,0,0,0,0,0,64.29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19461,47003828,47208,沖縄県,浦添市,牧港,None,NaN,４ＬＤＫ,80,平成31年,...,0,0,1,0,0,0,0,0,1,642.85
19462,47003829,47208,沖縄県,浦添市,牧港,None,NaN,２ＬＤＫ,70,平成10年,...,0,1,0,0,0,0,1,0,0,642.85
19463,47003880,47208,沖縄県,浦添市,港川,None,NaN,４ＬＤＫ,50,平成12年,...,0,0,1,0,0,0,0,0,1,642.85
19464,47006648,47211,沖縄県,沖縄市,与儀,None,NaN,３ＬＤＫ,60,平成31年,...,0,0,1,0,0,0,0,1,0,642.85


In [21]:
import pickle
print("\n--- KNN Imputationで欠損値補完（訓練時と同じ処理） ---")

# 保存したScalerとImputerを読み込み
with open('../models/imputation_scaler.pkl', 'rb') as f:
    imputation_scaler = pickle.load(f)

with open('../models/imputation_imputer.pkl', 'rb') as f:
    imputation_imputer = pickle.load(f)

with open('../models/imputation_cols.json', 'r', encoding='utf-8') as f:
    imputation_info = json.load(f)
    target_cols = imputation_info['target_cols']
    cols_to_impute = imputation_info['cols_to_impute']

# テストデータに適用（訓練時と完全に同じ処理）
X_test_impute = df_test[cols_to_impute].copy()

# Step 1: 訓練時に学習したScalerで標準化
X_test_scaled = imputation_scaler.transform(X_test_impute)

# Step 2: 訓練時に学習したImputerで補完
X_test_imputed_scaled = imputation_imputer.transform(X_test_scaled)

# Step 3: 逆変換して元のスケールに戻す
X_test_imputed = imputation_scaler.inverse_transform(X_test_imputed_scaled)

# 結果を書き戻し
df_test['取引時点での築年数'] = X_test_imputed[:, 0]
df_test['最寄駅：距離（分）'] = X_test_imputed[:, 1]

print(f"✅ テストデータの欠損値補完完了")
print(f"   築年数の平均: {df_test['取引時点での築年数'].mean():.2f}")
print(f"   駅距離の平均: {df_test['最寄駅：距離（分）'].mean():.2f}")
# ============================================



--- KNN Imputationで欠損値補完（訓練時と同じ処理） ---
✅ テストデータの欠損値補完完了
   築年数の平均: 22.31
   駅距離の平均: 9.00


In [8]:
import os
os.getcwd()
# The 'r' before the string makes it a "raw" string
os.chdir(r'C:\Users\sabri\Downloads\マンション価格予測\data')

#市区町村別の人口密度
df_mitudo = pd.read_csv('市区町村別人口密度.csv')
# 結合キーの列名
merge_key = '市区町村コード'

# 結合の実行
df_test = pd.merge(df_test, df_mitudo, on=merge_key, how='left')
df_test.info()
df_test

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19466 entries, 0 to 19465
Data columns (total 35 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                19466 non-null  int64  
 1   市区町村コード           19466 non-null  int64  
 2   都道府県名             19466 non-null  object 
 3   市区町村名             19466 non-null  object 
 4   地区名               19463 non-null  object 
 5   最寄駅：名称            19453 non-null  object 
 6   最寄駅：距離（分）         18509 non-null  float64
 7   間取り               18544 non-null  object 
 8   面積（㎡）             19466 non-null  int64  
 9   建築年               18804 non-null  object 
 10  建物の構造             18201 non-null  object 
 11  用途                13480 non-null  object 
 12  今後の利用目的           18439 non-null  object 
 13  都市計画              19122 non-null  object 
 14  建ぺい率（％）           19045 non-null  float64
 15  容積率（％）            19045 non-null  float64
 16  取引時点              19466 non-null  object

,ID,市区町村コード,都道府県名,市区町村名,地区名,最寄駅：名称,最寄駅：距離（分）,間取り,面積（㎡）,建築年,...,改装_改装済,改装_未改装,間取り_grouped_その他,間取り_grouped_１Ｋ,間取り_grouped_１ＬＤＫ,間取り_grouped_２ＬＤＫ,間取り_grouped_３ＬＤＫ,間取り_grouped_４ＬＤＫ,人口密度,市区町村人口密度
0,1000000,1101,北海道,札幌市中央区,旭ケ丘,円山公園,26.0,３ＬＤＫ,75,昭和64年,...,0,1,0,0,0,0,1,0,64.29,10660.3
1,1000056,1101,北海道,札幌市中央区,大通西,西１１丁目,1.0,２ＬＤＫ,55,平成28年,...,0,1,0,0,0,1,0,0,64.29,10660.3
2,1000108,1101,北海道,札幌市中央区,大通西,西１８丁目,2.0,１Ｒ,15,昭和64年,...,0,1,1,0,0,0,0,0,64.29,10660.3
3,1000109,1101,北海道,札幌市中央区,大通西,西１８丁目,2.0,１ＬＤＫ,45,平成3年,...,1,0,0,0,1,0,0,0,64.29,10660.3
4,1000110,1101,北海道,札幌市中央区,大通西,西１８丁目,3.0,１Ｒ,20,昭和56年,...,0,0,1,0,0,0,0,0,64.29,10660.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19461,47003828,47208,沖縄県,浦添市,牧港,None,NaN,４ＬＤＫ,80,平成31年,...,0,1,0,0,0,0,0,1,642.85,8721.4
19462,47003829,47208,沖縄県,浦添市,牧港,None,NaN,２ＬＤＫ,70,平成10年,...,1,0,0,0,0,1,0,0,642.85,8721.4
19463,47003880,47208,沖縄県,浦添市,港川,None,NaN,４ＬＤＫ,50,平成12年,...,0,1,0,0,0,0,0,1,642.85,8721.4
19464,47006648,47211,沖縄県,沖縄市,与儀,None,NaN,３ＬＤＫ,60,平成31年,...,0,1,0,0,0,0,1,0,642.85,6658.6


In [9]:
# '市区町村コード'を基準に重複している行を全て表示
df_crime_rate = pd.read_excel('市区町村別犯罪率.xlsx')
duplicates = df_crime_rate[df_crime_rate.duplicated(subset=['市区町村コード'], keep=False)]
print("--- 重複している犯罪率データ ---")
print(duplicates.sort_values(by='市区町村コード'))

# もし犯罪率が年によって異なるデータの場合、最新年を採用するか、平均を取るか決める必要があります。

--- 重複している犯罪率データ ---
     Unnamed: 0  犯罪発生率  市区町村コード
728        1020   0.81     1233
834        1307   0.61     1233
729        1021   0.81     7213
835        1308   0.61     7213
487         633   1.10    13206
847        1334   0.60    13206
246         285   1.55    22341
836        1309   0.61    22341
239         277   1.57    27381
507         662   1.08    27381
240         278   1.57    28464
508         663   1.08    28464
488         634   1.10    34208
848        1335   0.60    34208
459         587   1.14    38401
917        1860   0.23    38401


In [10]:
# 結合前に、'市区町村コード'ごとに重複している行を削除（最初の行を採用）
# これにより、df_crime_rateは結合キーでユニークになります。
df_crime_rate_unique = df_crime_rate.drop_duplicates(
    subset=['市区町村コード'], 
    keep='first'
)

print(f"✅ 犯罪率データの重複解消後の行数: {len(df_crime_rate_unique)}")

# 修正されたデータフレームを使ってマージを実行
df_test = pd.merge(df_test, df_crime_rate_unique, on=merge_key, how='left')

# 欠損値補完（この部分は変更なし）
mean_crime_rate = df_test['犯罪発生率'].mean()
df_test['犯罪発生率'] = df_test['犯罪発生率'].fillna(mean_crime_rate)

# マージ後の行数が、結合前のdf_testの行数と同じであることを確認してください

✅ 犯罪率データの重複解消後の行数: 919


In [11]:
df_additional=pd.read_csv('外部特徴量_2004-2019.csv')
merge_key = '取引時点_年'
df_test = pd.merge(df_test, df_additional, on=merge_key, how='left')
df_test

,ID,市区町村コード,都道府県名,市区町村名,地区名,最寄駅：名称,最寄駅：距離（分）,間取り,面積（㎡）,建築年,...,消費者物価指数(2010=100),失業率(%),日経平均株価(円・年末),10年国債利回り(%),USD/JPY(円),総人口,就業者数(百万人),人口増減率(%),高齢化率(65歳以上・%),世帯数(万世帯)
0,1000000,1101,北海道,札幌市中央区,旭ケ丘,円山公園,26.0,３ＬＤＫ,75,昭和64年,...,100.02,2.36,23656.62,-0.09,109.01,126.22,67.24,-0.22,28.4,"5,579"
1,1000056,1101,北海道,札幌市中央区,大通西,西１１丁目,1.0,２ＬＤＫ,55,平成28年,...,100.02,2.36,23656.62,-0.09,109.01,126.22,67.24,-0.22,28.4,"5,579"
2,1000108,1101,北海道,札幌市中央区,大通西,西１８丁目,2.0,１Ｒ,15,昭和64年,...,100.02,2.36,23656.62,-0.09,109.01,126.22,67.24,-0.22,28.4,"5,579"
3,1000109,1101,北海道,札幌市中央区,大通西,西１８丁目,2.0,１ＬＤＫ,45,平成3年,...,100.02,2.36,23656.62,-0.09,109.01,126.22,67.24,-0.22,28.4,"5,579"
4,1000110,1101,北海道,札幌市中央区,大通西,西１８丁目,3.0,１Ｒ,20,昭和56年,...,100.02,2.36,23656.62,-0.09,109.01,126.22,67.24,-0.22,28.4,"5,579"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19461,47003828,47208,沖縄県,浦添市,牧港,None,NaN,４ＬＤＫ,80,平成31年,...,100.02,2.36,23656.62,-0.09,109.01,126.22,67.24,-0.22,28.4,"5,579"
19462,47003829,47208,沖縄県,浦添市,牧港,None,NaN,２ＬＤＫ,70,平成10年,...,100.02,2.36,23656.62,-0.09,109.01,126.22,67.24,-0.22,28.4,"5,579"
19463,47003880,47208,沖縄県,浦添市,港川,None,NaN,４ＬＤＫ,50,平成12年,...,100.02,2.36,23656.62,-0.09,109.01,126.22,67.24,-0.22,28.4,"5,579"
19464,47006648,47211,沖縄県,沖縄市,与儀,None,NaN,３ＬＤＫ,60,平成31年,...,100.02,2.36,23656.62,-0.09,109.01,126.22,67.24,-0.22,28.4,"5,579"


In [12]:
df_test=df_test.drop('Unnamed: 0',axis=1)
df_test.columns

Index(['ID', '市区町村コード', '都道府県名', '市区町村名', '地区名', '最寄駅：名称', '最寄駅：距離（分）', '間取り',
       '面積（㎡）', '建築年', '建物の構造', '用途', '今後の利用目的', '都市計画', '建ぺい率（％）', '容積率（％）',
       '取引時点', '取引の事情等', '取引時点_年', '建築年_西暦', '築年数_欠損', '取引時点での築年数',
       '取引の事情等_調停・競売等', '取引の事情等_関係者間取引', '取引の事情等_その他', '改装_改装済', '改装_未改装',
       '間取り_grouped_その他', '間取り_grouped_１Ｋ', '間取り_grouped_１ＬＤＫ',
       '間取り_grouped_２ＬＤＫ', '間取り_grouped_３ＬＤＫ', '間取り_grouped_４ＬＤＫ', '人口密度',
       '市区町村人口密度', '犯罪発生率', '年', '名目GDP(兆円)', '実質GDP(兆円)', 'GDPデフレーター',
       '消費者物価指数(2010=100)', '失業率(%)', '日経平均株価(円・年末)', '10年国債利回り(%)',
       'USD/JPY(円)', '総人口', '就業者数(百万人)', '人口増減率(%)', '高齢化率(65歳以上・%)',
       '世帯数(万世帯)'],
      dtype='object')

In [13]:
import pandas as pd
import numpy as np
import warnings
import json
# import joblib # 訓練データからマップをロードする場合に必要

warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 3.8. 交互作用項と非線形変換の作成
# ------------------------------------------------------------------------------
print("\n" + "="*50)
print("3.8. 交互作用項と非線形変換を作成中...")
print("="*50)


# 計算に必要な外部データ列が存在するか確認し、NaNで初期化 (安全対策)
REQUIRED_COLS = ['取引時点での築年数', '面積（㎡）', '最寄駅：距離（分）', 
                 '建ぺい率（％）', '容積率（％）', '人口密度', '市区町村人口密度']
for col in REQUIRED_COLS:
    if col not in df_test.columns:
        df_test[col] = np.nan
        # print(f"   ⚠️ 警告: '{col}' が見つからないため、NaNで初期化しました。")


# 2. 交互作用項の作成
print("\n【非線形変換と交互作用項】")

# 築年数関連（非線形変換）
df_test['築年数_2乗'] = df_test['取引時点での築年数'] ** 2
df_test['築年数_3乗'] = df_test['取引時点での築年数'] ** 3
df_test['築年数_log'] = np.log1p(df_test['取引時点での築年数'])

# 面積関連（非線形変換）
df_test['面積_log'] = np.log1p(df_test['面積（㎡）'])
df_test['面積_平方根'] = np.sqrt(df_test['面積（㎡）'])

# 2項の交互作用
df_test['築年数×面積'] = df_test['取引時点での築年数'] * df_test['面積（㎡）']
df_test['築年数×駅距離'] = df_test['取引時点での築年数'] * df_test['最寄駅：距離（分）']
df_test['築年数×建ぺい率'] = df_test['取引時点での築年数'] * df_test['建ぺい率（％）']
df_test['築年数×容積率'] = df_test['取引時点での築年数'] * df_test['容積率（％）']
df_test['築年数×人口密度'] = df_test['取引時点での築年数'] * df_test['人口密度']
df_test['面積×駅距離'] = df_test['面積（㎡）'] * df_test['最寄駅：距離（分）']
df_test['面積×建ぺい率'] = df_test['面積（㎡）'] * df_test['建ぺい率（％）']
df_test['面積×容積率'] = df_test['面積（㎡）'] * df_test['容積率（％）']
df_test['面積×人口密度'] = df_test['面積（㎡）'] * df_test['人口密度']

# 建ぺい率・容積率の組み合わせ
df_test['建築可能性'] = df_test['建ぺい率（％）'] * df_test['容積率（％）'] / 100
# ゼロ割を避けるため、分母に0.1を加える
df_test['容積率_建ぺい率比'] = df_test['容積率（％）'] / (df_test['建ぺい率（％）'].replace(0, 0.1) + 0.1) 
df_test['建ぺい率_2乗'] = df_test['建ぺい率（％）'] ** 2
df_test['容積率_2乗'] = df_test['容積率（％）'] ** 2

# 駅距離関連
# ゼロ割を避けるため、分母に1を加える
df_test['駅距離_逆数'] = 1 / (df_test['最寄駅：距離（分）'] + 1) 
df_test['駅距離_log'] = np.log1p(df_test['最寄駅：距離（分）'])
df_test['駅距離_2乗'] = df_test['最寄駅：距離（分）'] ** 2
df_test['駅距離×建ぺい率'] = df_test['最寄駅：距離（分）'] * df_test['建ぺい率（％）']
df_test['駅距離×容積率'] = df_test['最寄駅：距離（分）'] * df_test['容積率（％）']

# 人口密度関連
df_test['人口密度_log'] = np.log1p(df_test['人口密度'])
df_test['市区町村人口密度_log'] = np.log1p(df_test['市区町村人口密度'])
df_test['人口密度×建ぺい率'] = df_test['人口密度'] * df_test['建ぺい率（％）']

# 3項の交互作用（試験的）
df_test['築年数×面積×駅距離'] = df_test['取引時点での築年数'] * df_test['面積（㎡）'] * df_test['最寄駅：距離（分）']
df_test['面積×建ぺい率×容積率'] = df_test['面積（㎡）'] * df_test['建ぺい率（％）'] * df_test['容積率（％）']
print("✓ 交互作用項の作成完了")

# ========================================
# 4. 頻度エンコーディング
# ========================================
# ========================================
# 推論時（テストデータ）用コード
# ========================================

# 保存した頻度マッピングを読み込み
print("\n" + "="*50)
print("頻度エンコーディング中（訓練データの頻度を使用）...")
print("="*50)

with open('../models/frequency_mappings.json', 'r', encoding='utf-8') as f:
    frequency_mappings = json.load(f)

# 建ぺい率・容積率の頻度（訓練データの頻度を使用）
building_ratio_freq = {float(k): v for k, v in frequency_mappings['建ぺい率_頻度'].items()}
df_test['建ぺい率_頻度'] = df['建ぺい率（％）'].map(building_ratio_freq)

floor_ratio_freq = {float(k): v for k, v in frequency_mappings['容積率_頻度'].items()}
df_test['容積率_頻度'] = df['容積率（％）'].map(floor_ratio_freq)
print("✓ 建ぺい率_頻度, 容積率_頻度")

# 市区町村の頻度（訓練データの頻度を使用）
city_freq = frequency_mappings['市区町村_頻度']
df_test['市区町村_頻度'] = df['市区町村名'].map(city_freq)
df_test['市区町村_頻度_log'] = np.log1p(df['市区町村_頻度'])
print("✓ 市区町村_頻度, 市区町村_頻度_log")

# 駅名の頻度（訓練データの頻度を使用）
station_freq = frequency_mappings['駅名_頻度']
df_test['駅名_頻度'] = df['最寄駅：名称'].map(station_freq)
df_test['駅名_頻度_log'] = np.log1p(df['駅名_頻度'])
print("✓ 駅名_頻度, 駅名_頻度_log")

# 地区名の頻度（訓練データの頻度を使用）
district_freq = frequency_mappings['地区名_頻度']
df_test['地区名_頻度'] = df['地区名'].map(district_freq)
df_test['地区名_頻度_log'] = np.log1p(df['地区名_頻度'])
print("✓ 地区名_頻度, 地区名_頻度_log")

print("✅ 訓練データの頻度を使用した頻度エンコーディング完了")


3.8. 交互作用項と非線形変換を作成中...

【非線形変換と交互作用項】
✓ 交互作用項の作成完了

頻度エンコーディング中（訓練データの頻度を使用）...
✓ 建ぺい率_頻度, 容積率_頻度
✓ 市区町村_頻度, 市区町村_頻度_log
✓ 駅名_頻度, 駅名_頻度_log
✓ 地区名_頻度, 地区名_頻度_log
✅ 訓練データの頻度を使用した頻度エンコーディング完了


In [14]:
# 頻度特徴量がNaNや0になっていないか
freq_cols = [col for col in df_test.columns if '_頻度' in col]
for col in freq_cols:
    nan_count = df_test[col].isna().sum()
    zero_count = (df_test[col] == 0).sum()
    print(f"{col}: NaN={nan_count}, Zero={zero_count}/{len(df_test)}")

建ぺい率_頻度: NaN=0, Zero=0/19466
容積率_頻度: NaN=0, Zero=0/19466
市区町村_頻度: NaN=0, Zero=0/19466
市区町村_頻度_log: NaN=0, Zero=0/19466
駅名_頻度: NaN=0, Zero=0/19466
駅名_頻度_log: NaN=0, Zero=0/19466
地区名_頻度: NaN=12, Zero=0/19466
地区名_頻度_log: NaN=12, Zero=0/19466


In [17]:
import pandas as pd
import os

print("\n" + "="*50)
print("4. 処理後のデータを保存")
print("="*50)

# --- 1. データフレームのサマリー表示 ---
print("--- 特徴量作成後のデータ (df_test) ---")
print(df_test.head())

# --- 2. 処理後のデータを保存 ---
# 予測用データは 'processed_test' ディレクトリに保存します。
PROCESSED_TEST_DIR = '../data/processed_test/'
PROCESSED_TEST_FILE = 'test_features.parquet' # モデル予測用の最終データ
PROCESSED_TEST_PATH = os.path.join(PROCESSED_TEST_DIR, PROCESSED_TEST_FILE)

# ディレクトリが存在しない場合は作成
os.makedirs(PROCESSED_TEST_DIR, exist_ok=True)

# Parquet形式で保存 (index=Falseでインデックスを保存しない)
df_test.to_parquet(PROCESSED_TEST_PATH, index=False)

print(f"\n✅ 特徴量作成後の予測用データを {PROCESSED_TEST_PATH} に保存しました。")
print(f"行数: {len(df_test)}, カラム数: {df_test.shape[1]}")


4. 処理後のデータを保存
--- 特徴量作成後のデータ (df_test) ---
        ID  市区町村コード 都道府県名   市区町村名  地区名 最寄駅：名称  最寄駅：距離（分）   間取り  面積（㎡）    建築年  \
0  1000000     1101   北海道  札幌市中央区  旭ケ丘   円山公園       26.0  ３ＬＤＫ     75  昭和64年   
1  1000056     1101   北海道  札幌市中央区  大通西  西１１丁目        1.0  ２ＬＤＫ     55  平成28年   
2  1000108     1101   北海道  札幌市中央区  大通西  西１８丁目        2.0    １Ｒ     15  昭和64年   
3  1000109     1101   北海道  札幌市中央区  大通西  西１８丁目        2.0  １ＬＤＫ     45   平成3年   
4  1000110     1101   北海道  札幌市中央区  大通西  西１８丁目        3.0    １Ｒ     20  昭和56年   

   ...  容積率_頻度 市区町村_頻度 市区町村_頻度_log 駅名_頻度  駅名_頻度_log  地区名_頻度 地区名_頻度_log  \
0  ...  261314    1160    7.057037   425   6.054439   322.0   5.777652   
1  ...   81310    5044    8.526153   427   6.059123   225.0   5.420535   
2  ...   81310    5044    8.526153   538   6.289716   180.0   5.198497   
3  ...   99463    1160    7.057037   177   5.181784    82.0   4.418841   
4  ...  261314    1160    7.057037   280   5.638355    44.0   3.806662   

  都市計画_高価格帯  都市計画_中価格帯  都市計画_低

In [19]:
df_test.columns

Index(['ID', '市区町村コード', '都道府県名', '市区町村名', '地区名', '最寄駅：名称', '最寄駅：距離（分）', '間取り',
       '面積（㎡）', '建築年', '建物の構造', '用途', '今後の利用目的', '都市計画', '建ぺい率（％）', '容積率（％）',
       '取引時点', '取引の事情等', '取引時点_年', '建築年_西暦', '築年数_欠損', '取引時点での築年数',
       '取引の事情等_調停・競売等', '取引の事情等_関係者間取引', '取引の事情等_その他', '改装_改装済', '改装_未改装',
       '間取り_grouped_その他', '間取り_grouped_１Ｋ', '間取り_grouped_１ＬＤＫ',
       '間取り_grouped_２ＬＤＫ', '間取り_grouped_３ＬＤＫ', '間取り_grouped_４ＬＤＫ', '人口密度',
       '市区町村人口密度', '犯罪発生率', '年', '名目GDP(兆円)', '実質GDP(兆円)', 'GDPデフレーター',
       '消費者物価指数(2010=100)', '失業率(%)', '日経平均株価(円・年末)', '10年国債利回り(%)',
       'USD/JPY(円)', '総人口', '就業者数(百万人)', '人口増減率(%)', '高齢化率(65歳以上・%)',
       '世帯数(万世帯)', '築年数_2乗', '築年数_3乗', '築年数_log', '面積_log', '面積_平方根', '築年数×面積',
       '築年数×駅距離', '築年数×建ぺい率', '築年数×容積率', '築年数×人口密度', '面積×駅距離', '面積×建ぺい率',
       '面積×容積率', '面積×人口密度', '建築可能性', '容積率_建ぺい率比', '建ぺい率_2乗', '容積率_2乗',
       '駅距離_逆数', '駅距離_log', '駅距離_2乗', '駅距離×建ぺい率', '駅距離×容積率', '人口密度_log',
       '市区町村人口密度_log', '人口密度×建ぺい率', '築年数×面積×駅距離', '面